![UTN Facultad Regional Mendoza](https://raw.githubusercontent.com/javovelez/Modelos-de-Lenguaje/main/img/logo_utn_frm.png)

# Laboratorio n° 2. Parte A: Las dos direcciones de una LSTM

**Asignatura:** Modelos de Lenguaje
**Bloque:** 2 — Redes Neuronales Recurrentes

---

## Introducción

Los tres modelos de la Unidad 2 comparten la misma capa. El clasificador de reseñas, el modelo de lenguaje y el encoder del traductor son todos una `nn.LSTM` que recorre la secuencia del primer token al último, un token por paso, actualizando un estado en cada uno. En los tres casos, cuando la capa llega al final, el estado resume lo que leyó.

Esa capa acepta un parámetro que todavía no usaste, `bidirectional=True`, y lo que hace es agregar una segunda pasada que recorre la misma secuencia del último token al primero. En este laboratorio vamos a explorar qué aporta ese cambio en dos tareas distintas: el traductor de la Clase 3 y el modelo de lenguaje de la Clase 2.

Al completar este laboratorio vas a poder:

- Implementar un encoder bidireccional y resolver la diferencia de formas que abre entre el estado que produce y el que el decoder espera.
- Verificar con una prueba propia que una operación sobre tensores hizo lo que esperabas, en un caso donde equivocarse no produce ningún error.
- Medir un traductor con BLEU global y por largo de la oración de entrada, y compararlo contra el traductor de la Clase 3 entrenado con los mismos datos y la misma configuración.
- Distinguir una mejora real de la dispersión entre corridas, entrenando cada configuración con más de una semilla.
- Interpretar una perplejidad contra las referencias medidas en clase, y localizar con una medición propia de dónde sale el valor obtenido.
- Explicar qué información tiene disponible un modelo en el momento de producir cada predicción, y en qué se diferencian las dos tareas en ese punto.

---

## Instrucciones generales

- Completá el código en las celdas marcadas con `# Tu código aquí`.
- Respondé las preguntas de análisis en las celdas de texto (tipo Markdown).
- **Este laboratorio necesita GPU.** En Colab: *Entorno de ejecución > Cambiar tipo de entorno > T4 GPU*. Entrena cuatro modelos —tres traductores de dos épocas y un modelo de lenguaje de mil pasos—. Medido en Colab, el notebook completo tarda unos 7 minutos con una T4 y 1 hora y 50 minutos sin acelerador.
- **Cada modelo entrenado se guarda en tu Drive.** La primera celda de preparación te va a pedir autorización para montarlo y escribe los checkpoints en `MyDrive/modelos-lenguaje-u2/`, una carpeta tuya que la celda crea si no existe. De eso se ocupa `entrenar_o_cargar`, el almacén de checkpoints de las clases. Si la sesión se corta, lo entrenado no se pierde, porque al volver a ejecutar se carga. **Autorizar el montaje no es obligatorio**, pero si lo rechazás los checkpoints quedan en el disco del entorno de ejecución, que se borra al reiniciarlo, y entonces hay que entrenar de nuevo.
- Este laboratorio va después de las Clases 2, 3 y 4, y da por vistos el modelo de lenguaje y su perplejidad (Clase 2), el encoder-decoder, el split por oración fuente y BLEU (Clase 3) y el cuello de botella del estado único (Clase 4). Las consignas te mandan a la sección correspondiente cuando hace falta.
- **La LSTM bidireccional no se vio en las clases.** La presenta el Ejercicio 1, con lo que hace falta para usarla.
- Las celdas de setup traen el código de las clases, sin cambios. Usalo sin modificarlo: si cambiás el corpus, el split o la configuración de entrenamiento, los números dejan de ser comparables con los de la clase y con los de tus compañeros.
- **Fijá las semillas que te pide cada enunciado.** El Ejercicio 3 se trata justamente de eso.
- Las cifras dependen de la máquina y del dispositivo. Lo que se corrige es la dirección del resultado y el argumento con el que lo explicás, no el tercer decimal.

## IMPORTANTE: qué celdas podés modificar

Este laboratorio es un **entregable**. Solo debés completar las celdas de actividad, que son las que aparecen con el comentario `# Tu código aquí` o el texto `*(Escribí tu respuesta acá)*`. Todas las demás celdas —enunciados, explicaciones, ejemplos provistos y encabezado— **no se tocan**.

La corrección se hace celda por celda: cada respuesta se busca en la celda donde el enunciado la pide. Si escribís en otro lado, o si movés, renombrás o borrás celdas del enunciado, esa parte de tu entrega queda sin poder corregirse.

Si querés probar algo suelto, hacelo en la misma celda de actividad o en una celda nueva que agregues, y borrala antes de entregar.

---
## Preparación

Las cuatro celdas que siguen ya vienen resueltas, pero **hay que ejecutarlas** en orden antes de empezar, y conviene leer las presentaciones, porque definen todos los nombres que usan los ejercicios.

La primera importa las librerías, baja los módulos de la materia —la clase `Vocabulario` y el almacén de checkpoints, con `entrenar_o_cargar`— y deja definidos `DISPOSITIVO`, `CHECKPOINTS`, `bajar` y `n_parametros`. Las tres siguientes arman la Sección A: el corpus de traducción, el modelo de la Clase 3 y el entrenamiento de ese modelo, que es el resultado contra el que vas a comparar el tuyo. La Sección B tiene sus dos celdas de preparación propias, más abajo.

**Al ejecutar la primera celda, Colab te va a pedir autorización para acceder a tu Drive.** Lo pide `preparar_carpeta()`, que monta tu Drive en el entorno de ejecución y crea ahí `MyDrive/modelos-lenguaje-u2/`, la carpeta donde queda un archivo `.pt` por cada uno de los cinco modelos del laboratorio. Es una carpeta tuya, no de la cátedra, y es la misma que usan los notebooks de teoría de la unidad, así que si corriste el de las Clases 3 y 4 el traductor de la clase ya está guardado ahí y la preparación se ahorra la descarga.

Autorizar el montaje no es obligatorio. Si lo rechazás, o si ejecutás el notebook fuera de Colab, `preparar_carpeta()` usa una carpeta `modelos-lenguaje-u2` en el disco del entorno de ejecución, que se borra cuando el entorno se reinicia. El laboratorio corre igual, pero después de cada reinicio tenés que volver a bajar la línea de base y a entrenar los cuatro modelos. La celda imprime una línea que empieza con `checkpoints en:` y sigue con la ruta que quedó, y ahí podés ver cuál de las dos carpetas usó. Si la ruta empieza con `/content/drive/`, los checkpoints están en tu Drive y sobreviven al corte de la sesión.

In [ ]:
# ─── Setup: imports, módulos de la materia y carpeta de checkpoints ─────────
import os, math, time, random, urllib.request
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence

try:
    import sacrebleu
except ImportError:
    os.system("pip install -q sacrebleu")
    import sacrebleu

# El módulo con la clase Vocabulario es el del Laboratorio 1a, el mismo de las clases.
URL_PIPELINE = ("https://github.com/javovelez/Modelos-de-Lenguaje/releases/download"
                "/utils-v1/lab1a_pipeline.py")
if not os.path.exists("lab1a_pipeline.py"):
    urllib.request.urlretrieve(URL_PIPELINE, "lab1a_pipeline.py")

# Los tres archivos se bajan del repositorio de la materia, que da una URL estable.
# El Drive de la cátedra queda de respaldo, con los mismos identificadores que usan
# los notebooks de las clases: es la misma copia, pero Drive corta las descargas
# públicas por cuota cuando muchos bajan el mismo archivo a la vez.
RELEASES = "https://github.com/javovelez/Modelos-de-Lenguaje/releases/download"
FUENTES = {                                   # archivo: (URL del release, ID de Drive)
    "cache_modelos.py":       (f"{RELEASES}/utils-v1/cache_modelos.py",
                               "1k5wxbSMS-_3dC8FN_F-2VN6dp8tGFCpz"),
    "tatoeba_es_en.txt":      (f"{RELEASES}/corpus-u2-v1/tatoeba_es_en.txt",
                               "19FEvckK9Mj9WPEsS3jqWuCIEf8cWpLmM"),
    "cuentos_es_top5000.txt": (f"{RELEASES}/corpus-u2-v1/cuentos_es_top5000.txt",
                               "1hIeUjBuvZrNC9d0xds9vRVOky52a_0oM"),
    "traductor.pt":           (f"{RELEASES}/utils-v1/traductor.pt", None),
}


def bajar(nombre, destino=None):
    """Baja un archivo del repositorio de la materia; si no responde, del Drive de la cátedra."""
    destino = destino or nombre
    if os.path.exists(destino):
        return
    url, id_drive = FUENTES[nombre]
    try:
        urllib.request.urlretrieve(url, destino)
    except Exception as falla:
        if id_drive is None:                  # sin copia en el Drive de la cátedra
            print(f"no se pudo bajar {nombre} ({falla}).")
            return
        print(f"el repositorio de la materia no respondió ({falla}); se usa el Drive.")
        try:
            import gdown
        except ImportError:
            os.system("pip install -q gdown")
            import gdown
        gdown.download(id=id_drive, output=destino, quiet=False)


bajar("cache_modelos.py")

from lab1a_pipeline import Vocabulario
from cache_modelos import preparar_carpeta, entrenar_o_cargar

torch.manual_seed(0)
random.seed(0)
np.random.seed(0)

DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
CHECKPOINTS = preparar_carpeta()  # la misma carpeta de checkpoints que las clases


def n_parametros(m):
    """Cantidad total de parámetros de un modelo."""
    return sum(p.numel() for p in m.parameters())


pd.Series({"torch": torch.__version__, "sacrebleu": sacrebleu.__version__,
           "dispositivo": DISPOSITIVO})

### El corpus de traducción, el split y los dos vocabularios

La celda que sigue es la Parte 1 y la Parte 2 de la Clase 3, sin cambios: el archivo de pares español-inglés de Tatoeba, el split **por oración fuente** —cada oración en español cae entera de un solo lado, para que ninguna traducción del conjunto de prueba se haya visto al entrenar— y un vocabulario por idioma construido solo con entrenamiento.

Deja definidos, entre otros:

| Nombre | Qué es |
|---|---|
| `pares_ent`, `pares_val` | pares `(español, inglés)` de entrenamiento y validación |
| `prueba` | 3.000 oraciones en español, cada una con todas sus traducciones de referencia |
| `ORACIONES_PRUEBA`, `REFERENCIAS_PRUEBA` | las dos columnas de `prueba` por separado, que es como las pide BLEU |
| `vocab_es`, `vocab_en`, `V_ES`, `V_EN` | los dos vocabularios y sus tamaños |
| `ParesDeTraduccion`, `loaders` | el `Dataset` de pares y la función que arma los dos `DataLoader` |
| `dl_ent`, `dl_val` | los `DataLoader` de entrenamiento y validación, con la entrada invertida |
| `DIM`, `OCULTA`, `LOTE`, `INVERTIR` | la configuración de la clase: 128, 256, 64 y `True` |

`INVERTIR` activa la inversión de la entrada de la Clase 3, sección *3.1*, que le pasa al encoder la oración al revés para que sus primeras palabras queden cerca del comienzo de la decodificación. `loaders(invertir)` permite armar los `DataLoader` con o sin esa inversión, y el Ejercicio 3 la usa.

In [ ]:
# ─── Setup: el corpus de traducción, el split y los dos vocabularios ────────
bajar("tatoeba_es_en.txt")

pares = [tuple(linea.rstrip("\n").split(" -> "))
         for linea in open("tatoeba_es_en.txt", encoding="utf-8")]
por_fuente = defaultdict(list)
for es, en in pares:
    por_fuente[es].append(en)

# Split por oración FUENTE (Clase 3, sección 2.2): la misma oración en español no
# puede estar de los dos lados, aunque tenga varias traducciones distintas.
fuentes = list(por_fuente)
perm_f = torch.randperm(len(fuentes), generator=torch.Generator().manual_seed(0)).tolist()
N_FUENTES = 3_000
f_prueba = [fuentes[i] for i in perm_f[:N_FUENTES]]
f_val    = [fuentes[i] for i in perm_f[N_FUENTES:2 * N_FUENTES]]
f_ent    = [fuentes[i] for i in perm_f[2 * N_FUENTES:]]

pares_ent = [(es, en) for es in f_ent for en in por_fuente[es]]
pares_val = [(es, en) for es in f_val for en in por_fuente[es]]
prueba    = [(es, por_fuente[es]) for es in f_prueba]     # (oración, [referencias])

ORACIONES_PRUEBA   = [es for es, _ in prueba]
REFERENCIAS_PRUEBA = [refs for _, refs in prueba]

# Los vocabularios se construyen SOLO con entrenamiento (Clase 3, sección 2.3).
vocab_es = Vocabulario([f"{es} <eos>" for es, _ in pares_ent],
                       tokenizador=str.split, freq_min=2)
vocab_en = Vocabulario([f"<sos> {en} <eos>" for _, en in pares_ent],
                       tokenizador=str.split, freq_min=2)
V_ES, V_EN = len(vocab_es), len(vocab_en)

DIM, OCULTA, LOTE = 128, 256, 64
INVERTIR = True


class ParesDeTraduccion(Dataset):
    """(entrada con <eos>, opcionalmente invertida) y (salida entre <sos> y <eos>)."""

    def __init__(self, pares, invertir=INVERTIR):
        self.pares, self.invertir = pares, invertir

    def __len__(self):
        return len(self.pares)

    def __getitem__(self, i):
        es, en = self.pares[i]
        entrada = vocab_es.codificar(f"{es} <eos>")
        if self.invertir:
            entrada = entrada[::-1]
        salida = vocab_en.codificar(f"<sos> {en} <eos>")
        return torch.tensor(entrada), torch.tensor(salida)


def armar_lote(items):
    """Rellena con <pad> hasta la oración más larga de cada lado del lote."""
    entradas, salidas = zip(*items)
    return pad_sequence(entradas, batch_first=True), pad_sequence(salidas, batch_first=True)


def loaders(invertir=INVERTIR):
    dl_e = DataLoader(ParesDeTraduccion(pares_ent, invertir), batch_size=LOTE,
                      shuffle=True, drop_last=True, collate_fn=armar_lote)
    dl_v = DataLoader(ParesDeTraduccion(pares_val, invertir), batch_size=256,
                      collate_fn=armar_lote)
    return dl_e, dl_v


dl_ent, dl_val = loaders()

pd.Series({"pares de entrenamiento": len(pares_ent), "pares de validación": len(pares_val),
           "oraciones de prueba": len(prueba), "vocabulario español": V_ES,
           "vocabulario inglés": V_EN, "lotes por época": len(dl_ent)})

### El traductor de la Clase 3, su entrenamiento y BLEU

La celda que sigue trae el modelo de la Clase 3 y todo lo que hace falta para entrenarlo y evaluarlo, también sin cambios.

| Nombre | Qué es |
|---|---|
| `Traductor` | encoder LSTM, estado final, decoder LSTM y `Linear(V_EN)` en cada posición. Su método `codificar(src)` devuelve el estado inicial `(h, c)` del decoder |
| `perdida_traduccion` | entropía cruzada por token, ignorando las posiciones de relleno |
| `codificar_oraciones` | codifica una lista de oraciones en español, las rellena y las deja en el dispositivo |
| `traducir` | traducción voraz por lotes, hasta `<eos>` o 12 tokens |
| `bleu`, `bleu_por_largo` | BLEU de corpus con `sacrebleu`, global y separado en cuatro tramos de largo de la oración de entrada |
| `entrenador_de` | arma el entrenador para un par de `DataLoader` dado, con la configuración de la clase |
| `EPOCAS`, `LR`, `CADA` | 2 épocas, `lr` 2e-3, y una medición cada 250 pasos |

El método que importa para el Ejercicio 1 es **`codificar`**, el único que la clase del ejercicio va a redefinir. Todo lo demás del `Traductor` —el decoder, la capa de salida, el `forward`— queda igual.

In [ ]:
# ─── Setup: el traductor de la Clase 3, su entrenamiento y BLEU ─────────────
class Traductor(nn.Module):
    """Encoder LSTM → estado final → decoder LSTM → Linear(V_EN) en cada posición."""

    def __init__(self, n_src=None, n_tgt=None, dim=DIM, oculta=OCULTA):
        super().__init__()
        n_src, n_tgt = n_src or V_ES, n_tgt or V_EN
        self.emb_src = nn.Embedding(n_src, dim, padding_idx=0)
        self.emb_tgt = nn.Embedding(n_tgt, dim, padding_idx=0)
        self.encoder = nn.LSTM(dim, oculta, batch_first=True)
        self.decoder = nn.LSTM(dim, oculta, batch_first=True)
        self.salida  = nn.Linear(oculta, n_tgt)

    def codificar(self, src):
        """El estado inicial (h, c) del decoder, sin contar el relleno."""
        largos = (src != 0).sum(1).cpu()
        empaquetada = pack_padded_sequence(self.emb_src(src), largos,
                                           batch_first=True, enforce_sorted=False)
        _, estado = self.encoder(empaquetada)
        return estado

    def forward(self, src, tgt_entrada):
        estado = self.codificar(src)
        h, _ = self.decoder(self.emb_tgt(tgt_entrada), estado)
        return self.salida(h)


def perdida_traduccion(logits, y):
    """Entropía cruzada media por token, ignorando las posiciones de <pad>."""
    return F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1), ignore_index=0)


# ─── Traducción voraz (Clase 3, sección 4.1) ────────────────────────────────
SOS, EOS = vocab_en["<sos>"], vocab_en["<eos>"]


def codificar_oraciones(oraciones, invertir=INVERTIR):
    ds = ParesDeTraduccion([(es, "") for es in oraciones], invertir)
    return pad_sequence([ds[i][0] for i in range(len(oraciones))], batch_first=True).to(DISPOSITIVO)


@torch.no_grad()
def traducir_lote(m, src, max_len=12):
    estado = m.codificar(src)
    token = torch.full((src.size(0), 1), SOS, dtype=torch.long, device=DISPOSITIVO)
    generado = []
    for _ in range(max_len):
        h, estado = m.decoder(m.emb_tgt(token), estado)
        token = m.salida(h).argmax(-1)
        generado.append(token)
    return torch.cat(generado, 1)


@torch.no_grad()
def traducir(m, oraciones, max_len=12, lote=256, invertir=INVERTIR):
    m.eval()
    salidas = []
    for i in range(0, len(oraciones), lote):
        src = codificar_oraciones(oraciones[i:i + lote], invertir)
        for fila in traducir_lote(m, src, max_len).tolist():
            fila = fila[:fila.index(EOS)] if EOS in fila else fila
            salidas.append(" ".join(vocab_en.decodificar(fila)))
    m.train()
    return salidas


# ─── BLEU (Clase 3, sección 5.1) ────────────────────────────────────────────
def bleu(hipotesis, referencias):
    """BLEU de corpus con varias referencias por oración."""
    n = max(len(r) for r in referencias)
    columnas = [[r[i] if i < len(r) else r[0] for r in referencias] for i in range(n)]
    return sacrebleu.corpus_bleu(hipotesis, columnas, tokenize="none")


largo_prueba = [len(es.split()) for es in ORACIONES_PRUEBA]
TRAMOS = {"2-3": (2, 3), "4-5": (4, 5), "6-7": (6, 7), "8": (8, 8)}


def bleu_por_largo(hipotesis):
    """BLEU separado en cuatro tramos, según cuántas palabras tiene la oración de entrada."""
    filas = {}
    for grupo, (a, b) in TRAMOS.items():
        idx = [i for i, l in enumerate(largo_prueba) if a <= l <= b]
        filas[grupo] = round(bleu([hipotesis[i] for i in idx],
                                  [REFERENCIAS_PRUEBA[i] for i in idx]).score, 1)
    return filas


# ─── El entrenador (Clase 3, sección 4.2) ───────────────────────────────────
EPOCAS, LR, CADA = 2, 2e-3, 250
EJEMPLOS = [es for es, _ in pares_val[:3]]


@torch.no_grad()
def perdida_en(m, dl):
    m.eval()
    total, n = 0.0, 0
    for src, tgt in dl:
        src, tgt = src.to(DISPOSITIVO), tgt.to(DISPOSITIVO)
        reales = (tgt[:, 1:] != 0).sum().item()
        total += perdida_traduccion(m(src, tgt[:, :-1]), tgt[:, 1:]).item() * reales
        n += reales
    m.train()
    return total / n


def entrenador_de(dl, dl_v=None, invertir=INVERTIR):
    """Devuelve el entrenador que `entrenar_o_cargar` le aplica al modelo."""
    dl_v = dl_val if dl_v is None else dl_v

    def entrenador(m):
        opt = torch.optim.Adam(m.parameters(), lr=LR)
        historia, muestras = [], {}
        paso, acumulada = 0, 0.0
        for epoca in range(EPOCAS):
            m.train()
            for src, tgt in dl:
                src, tgt = src.to(DISPOSITIVO), tgt.to(DISPOSITIVO)
                perdida = perdida_traduccion(m(src, tgt[:, :-1]), tgt[:, 1:])
                opt.zero_grad()
                perdida.backward()
                nn.utils.clip_grad_norm_(m.parameters(), 1.0)
                opt.step()
                paso += 1
                acumulada += perdida.item()
                if paso % CADA == 0:
                    val = perdida_en(m, dl_v)
                    muestras[paso] = traducir(m, EJEMPLOS, invertir=invertir)
                    historia.append({"paso": paso, "perdida_ent": acumulada / CADA,
                                     "perdida_val": val})
                    print(f"   paso {paso:5d}  pérdida ent {acumulada / CADA:.3f}  "
                          f"val {val:.3f} | {muestras[paso][0]}", flush=True)
                    acumulada = 0.0
        return {"historia": historia, "muestras": muestras}

    return entrenador


print(f"traductor de la clase: {n_parametros(Traductor()):,} parámetros")

### La línea de base: el traductor de la Clase 3

Los Ejercicios 2 y 3 miden el modelo que vas a escribir comparándolo contra el traductor de la Clase 3 sin modificar. Ese traductor es la **línea de base** del laboratorio, el resultado ya conocido contra el que se decide si un cambio de arquitectura aporta algo. Sin él, el BLEU del modelo nuevo es un número suelto y no hay con qué decidir si es alto o bajo.

**Este modelo no lo entrenás.** La celda que sigue baja el traductor de la Clase 3 ya entrenado del repositorio de la materia, lo deja en la carpeta de checkpoints y mide su BLEU sobre las 3.000 oraciones de prueba. Son 29 MB, y se bajan una sola vez: si el archivo ya está en la carpeta —porque corriste el notebook de teoría de las Clases 3 y 4, o porque ya ejecutaste esta celda antes—, no se vuelve a bajar.

`entrenar_o_cargar` verifica la **firma** del checkpoint contra la configuración que le pasa la celda, que es la exacta de la Clase 3. Si coincide, carga los pesos y no entrena nada. Entrenarlo acá costaría dos épocas, que sin acelerador son algo más de media hora, y no aportan nada al laboratorio, porque este traductor es el punto de comparación y no el objeto de ningún ejercicio.

La medición sí se hace en tu máquina, y por eso está en la preparación y no en un ejercicio: **todas las comparaciones del laboratorio tienen que salir de la misma sesión.**

Deja definidos `traductor`, `extras_base`, `bleu_base` y `largo_base`, que son el modelo, el registro de su entrenamiento, su BLEU global y su BLEU por tramo de largo.

In [ ]:
# ─── Setup: la línea de base, el traductor de la Clase 3 ────────────────────
# Este modelo no lo escribís vos, así que viene entrenado: se baja a la carpeta de
# checkpoints y entrenar_o_cargar lo encuentra ahí. Si ya está, no se baja de nuevo.
bajar("traductor.pt", os.path.join(CHECKPOINTS, "traductor.pt"))


def constructor_base():
    torch.manual_seed(0)
    return Traductor().to(DISPOSITIVO)


traductor, extras_base, _ = entrenar_o_cargar(
    "traductor", constructor_base, entrenador_de(dl_ent), vocab_en,
    vocab_es=V_ES, arq="Traductor", dim=DIM, oculta=OCULTA, invertir=INVERTIR,
    epocas=EPOCAS, lote=LOTE, lr=LR, n_ent=len(pares_ent), seed=0)
traductor = traductor.to(DISPOSITIVO)

hipotesis_base = traducir(traductor, ORACIONES_PRUEBA)
bleu_base  = bleu(hipotesis_base, REFERENCIAS_PRUEBA).score
largo_base = bleu_por_largo(hipotesis_base)

print(f"\nparámetros: {n_parametros(traductor):,}")
print(f"BLEU global: {bleu_base:.1f}")
print(f"BLEU por largo: {largo_base}")
pd.DataFrame({"oración": ORACIONES_PRUEBA[:5], "traducción": hipotesis_base[:5],
              "referencia": [r[0] for r in REFERENCIAS_PRUEBA[:5]]})

---
## Sección A: El encoder que lee en los dos sentidos

El traductor de la Clase 3 comprime la oración en español en un único vector de 256 números —el estado final del encoder— y el decoder produce la traducción entera a partir de ese vector. La Clase 4 midió el problema de ese diseño y lo resolvió con atención.

Esta sección aborda el mismo problema sin tocar el decoder, **mejorando el resumen** en vez de reemplazarlo.

### Ejercicio 1 — Un encoder bidireccional, y la forma del estado final

**Objetivo:** Implementar un encoder que lea la oración en los dos sentidos, resolver la diferencia de formas que eso abre con el decoder, y verificar con una prueba propia que la operación hizo lo que esperabas.

**Enunciado:**

> **Nota — qué hace `bidirectional=True`.** Una `nn.LSTM` creada con `bidirectional=True` no es una celda distinta: son **dos** LSTM con parámetros propios, una que recorre la secuencia de principio a fin y otra que la recorre de fin a principio. Eso cambia dos cosas. La salida en cada posición pasa a medir el doble de unidades, porque se concatena lo que calculó cada pasada. Y el estado final deja de ser uno solo: `h` y `c` pasan a tener forma `(2, B, unidades_por_direccion)`, donde el índice `0` de la primera dimensión es el estado con el que terminó la pasada de ida y el índice `1` el de la pasada de vuelta. Las dos leyeron la oración entera; lo que cambia es dónde terminaron.

1. **Escribí la clase `TraductorBidireccional`**, que herede de `Traductor` y reciba los mismos argumentos. Lo único que cambia en su `__init__` es `self.encoder`, que pasa a ser una `nn.LSTM` bidireccional con `OCULTA // 2` unidades **por dirección**. Con esa mitad, los dos estados finales concatenados miden exactamente `OCULTA` y el decoder los recibe sin ninguna capa intermedia.

2. **Escribí el método `codificar(self, src)`** de esa clase, que devuelva el par `(h, c)`, cada uno con forma `(1, B, OCULTA)`, donde la fila de cada oración del lote son su estado de ida y su estado de vuelta, uno a continuación del otro. La entrada se empaqueta con `pack_padded_sequence` igual que en `Traductor.codificar`, para que el relleno no participe (Clase 3, sección *3.2*).

3. **Creá el modelo.** Fijá la semilla de PyTorch en 0, creá una instancia de la clase en una variable llamada `trad_bidi`, pasala a `DISPOSITIVO` e imprimí su cantidad de parámetros junto a la del traductor de la clase.

4. **Verificá que el estado de cada oración es el suyo.** Codificá con `trad_bidi` un lote formado por las dos primeras oraciones en español de `pares_val`, y por separado cada una de esas dos oraciones sola. Comprobá con `torch.allclose` que el `h` que le toca a cada oración es el mismo en los dos casos, e imprimí las dos comparaciones.

> **Nota:** pasar de `(2, B, OCULTA // 2)` a `(1, B, OCULTA)` se puede escribir de varias maneras, y algunas devuelven un tensor **de la forma correcta con los valores de otra oración adentro**. Eso no produce ningún error, el entrenamiento corre igual y el modelo traduce peor. La comparación del punto 4 es lo único que lo detecta.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

a) El encoder devuelve un tensor de forma `(2, B, 128)` y el decoder espera uno de forma `(1, B, 256)`. Explicá qué representa cada uno de los tres ejes del tensor que devuelve el encoder, y por qué pasar de una forma a la otra con un `reshape` directo, sin reordenar los ejes antes, produce un tensor de la forma correcta con los valores equivocados.

b) `trad_bidi` tiene **menos** parámetros que el traductor de la clase, y sin embargo el decoder sigue recibiendo un estado del mismo tamaño. Explicá de dónde sale esa diferencia, con la cuenta que la justifica.

*(Escribí tu respuesta acá)*

### Ejercicio 2 — Entrenar el bidireccional y medirlo contra la clase

**Objetivo:** Entrenar el modelo del ejercicio anterior con la misma configuración con la que se entrenó la línea de base y compararlo con BLEU global y por largo de la oración de entrada.

**Enunciado:**

1. **Entrená `trad_bidi`** con `entrenar_o_cargar`, bajo el nombre `"lab2_trad_bidi"`. El segundo argumento de `entrenar_o_cargar` es el **constructor**, que acá no es el `__init__` de la clase sino una función sin argumentos que devuelve el modelo sin entrenar, igual que la `constructor()` de la Clase 3, sección *4.2*. El tuyo tiene que fijar la semilla de PyTorch en 0 y devolver una instancia nueva de `TraductorBidireccional` en `DISPOSITIVO`. Pasale el entrenador de la clase, `vocab_en`, y la misma configuración que usó la línea de base —`dim`, `oculta`, `invertir`, `epocas`, `lote`, `lr`, `n_ent` y `seed`—, con `arq="TraductorBidireccional"`. Son dos épocas. Repetir esa configuración es lo que deja al encoder como única diferencia entre los dos modelos: si cambiaras las épocas o el `lr`, la diferencia de BLEU ya no se podría atribuir a las dos direcciones.

2. **Medilo.** Traducí las 3.000 oraciones de `ORACIONES_PRUEBA` con `traducir`, que recibe el modelo y la lista de oraciones y devuelve la lista de traducciones, y sobre esa lista calculá el BLEU global con `bleu` y el BLEU por tramo de largo con `bleu_por_largo`. Guardá esos dos valores en `bleu_bidi` y `largo_bidi`, que el Ejercicio 3 vuelve a usar. Las tres funciones quedaron definidas en la preparación, y la celda de la línea de base las llama en ese mismo orden sobre el traductor de la clase. Armá después un `DataFrame` que compare el traductor de la clase con el bidireccional en siete filas: cantidad de parámetros, pérdida de validación al final del entrenamiento, BLEU global y BLEU en cada uno de los cuatro tramos de largo.

> **Nota:** `entrenar_o_cargar` calcula una firma con la configuración que le pasás. Si dos modelos distintos se guardan con la misma configuración y el mismo nombre, el segundo carga el primero; por eso cada modelo del laboratorio lleva su propio nombre y declara su `arq`.

> **Antes de ejecutar, verificá que tenés GPU.** Este es el primer modelo que entrenás en el laboratorio, y con una T4 tarda alrededor de un minuto, mientras que sin acelerador tarda alrededor de media hora. El Ejercicio 3 entrena otros dos de la misma duración. Si `DISPOSITIVO` no dice `cuda`, cambiá el entorno en *Entorno de ejecución > Cambiar tipo de entorno > T4 GPU*.

> **Pista:** la pérdida de validación de cada entrenamiento está en `extras["historia"][-1]["perdida_val"]`, y la de la línea de base quedó en `extras_base`.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

Respondé con la tabla del punto 2 a la vista.

a) Explicá qué información tiene el estado inicial del decoder que no tenía en el traductor de la clase.

b) El traductor de la clase le pasa al encoder la oración al revés (`INVERTIR = True`, Clase 3 sección *3.1*). Explicá qué problema busca resolver esa inversión, y por qué un encoder bidireccional la vuelve redundante.

*(Escribí tu respuesta acá)*

### Ejercicio 3 — ¿La diferencia es real?

**Objetivo:** Medir cuánto se mueve el resultado entre corridas idénticas salvo por la semilla, y usar esa cifra para decidir si una diferencia observada es un resultado o ruido.

**Enunciado:**

La comparación del Ejercicio 2 apoya toda la conclusión en dos números, `bleu_base` y `bleu_bidi`, y cada uno de los dos salió de **una** corrida, con la semilla 0. Dos modelos con la misma arquitectura, los mismos datos y la misma configuración, entrenados con semillas distintas, no dan el mismo resultado: cambian los valores iniciales de los pesos y el orden en que el `DataLoader` recorre los lotes. Antes de atribuirle una diferencia al cambio de arquitectura hay que saber cuánto se mueve el BLEU **sin** cambiar la arquitectura. Para eso vas a entrenar una segunda corrida de cada una de las dos arquitecturas, con la semilla 1, y a poner las cuatro juntas.

**Parte A — La línea de base, otra vez.**

1. Entrená el traductor de la clase con la **semilla 1**, bajo el nombre `"lab2_trad_base_s1"`. El modelo es `Traductor`, la clase que dejó definida la preparación, y la instanciás igual que instanciaste `TraductorBidireccional` en el Ejercicio 1. La celda de la línea de base te muestra el constructor con el que se la entrenó, y la única diferencia con esa celda es la semilla, que va fijada en el constructor y declarada en la configuración (`seed=1`), para que `entrenar_o_cargar` no confunda este modelo con el otro. Medí su BLEU global y por largo, y guardalos en `bleu_base_s1` y `largo_base_s1`.

In [ ]:
# Tu código aquí

**Parte B — El bidireccional, otra vez.**

2. Entrená el bidireccional —tu `TraductorBidireccional`— con la **semilla 1**, bajo el nombre `"lab2_trad_bidi_s1"`, con el mismo criterio, y guardá su BLEU global en `bleu_bidi_s1`.

3. **Poné las cuatro corridas juntas.** Imprimí una tabla de las dos arquitecturas por las dos semillas con el BLEU global de cada corrida, que son los cuatro números que ya tenés en memoria —`bleu_base` y `bleu_bidi` con la semilla 0, `bleu_base_s1` y `bleu_bidi_s1` con la semilla 1—, y agregale una columna con la diferencia entre las dos semillas de cada arquitectura.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

a) Con la tabla de las cuatro corridas, decidí si la ventaja del encoder bidireccional es un resultado o una casualidad, y justificá con las cifras que lo sostienen.

b) La tabla que sigue mide el efecto de apagar la inversión de la entrada, con **una** corrida por configuración, en la máquina de la cátedra:

| | invertir | BLEU |
|---|---|---|
| traductor de la clase | sí | 25,8 |
| traductor de la clase | no | 26,8 |
| bidireccional | sí | 29,6 |
| bidireccional | no | 28,4 |

Decí qué se puede concluir de esa tabla sobre si sigue haciendo falta invertir la entrada, y qué habría que medir para poder concluir algo.

*(Escribí tu respuesta acá)*

---
## Sección B: El mismo cambio, sobre el modelo de lenguaje

En la Sección A le agregaste la segunda dirección al encoder de un traductor. El modelo de lenguaje de la Clase 2 tiene una LSTM sobre el mismo idioma, y `nn.LSTM` acepta ahí el mismo `bidirectional=True`.

Esta sección hace ese cambio y lo mide. Entrena el modelo de la Clase 2 con la LSTM bidireccional sobre el mismo corpus de cuentos y compara su perplejidad en validación con las que la clase reportó.

### Preparación de la Sección B: el corpus de cuentos y las ventanas

Las dos celdas que siguen arman el modelo de lenguaje de la Clase 2, también sin cambios. La primera trae el corpus de cuentos recortado a las 5.000 palabras más frecuentes, el split por oración y las ventanas de 32 tokens; la segunda, el modelo y sus mediciones.

**Los nombres llevan el sufijo `_LM`** —`DIM_LM`, `OCULTA_LM`, `LOTE_LM`, `L_LM`, `vocab_lm`— aunque en la Clase 2 no lo llevan. El motivo es que la Sección A sigue viva en el mismo kernel y usa varios de esos nombres con otros valores; con el sufijo, volver a ejecutar cualquier celda de la Sección A después de haber corrido ésta sigue funcionando.

| Nombre | Qué es |
|---|---|
| `vocab_lm`, `V_LM` | el vocabulario de los cuentos y su tamaño |
| `ids_ent`, `ids_val` | el corpus entero como una secuencia de índices, entrenamiento y validación |
| `VentanasDeTexto` | el `Dataset` de ventanas de `L_LM + 1` tokens: `x` son los primeros `L_LM`, `y` los últimos |
| `dl_lm` | el `DataLoader` de entrenamiento |
| `L_LM`, `LOTE_LM` | 32 y 64 |

In [ ]:
# ─── Setup: el corpus de cuentos, el vocabulario y las ventanas ─────────────
bajar("cuentos_es_top5000.txt")

texto = open("cuentos_es_top5000.txt", encoding="utf-8").read()
oraciones = [o.rstrip(" .") + " ." for o in texto.split(" . ")]
vocab_lm = Vocabulario(oraciones, tokenizador=str.split)
V_LM = len(vocab_lm)

# Split por oración, y después todo pegado en una sola secuencia (Clase 2, sección 2.2).
perm = torch.randperm(len(oraciones), generator=torch.Generator().manual_seed(0))
N_VAL = len(oraciones) // 20
oraciones_val = [oraciones[i] for i in perm[:N_VAL]]
oraciones_ent = [oraciones[i] for i in perm[N_VAL:]]
ids_ent = torch.tensor([vocab_lm[t] for o in oraciones_ent for t in o.split()])
ids_val = torch.tensor([vocab_lm[t] for o in oraciones_val for t in o.split()])

L_LM, LOTE_LM = 32, 64


class VentanasDeTexto(Dataset):
    """Ventanas consecutivas de L+1 tokens: x son los primeros L, y los últimos L."""

    def __init__(self, ids, L):
        self.ids, self.L = ids, L

    def __len__(self):
        return (len(self.ids) - 1) // self.L

    def __getitem__(self, i):
        ventana = self.ids[i * self.L: i * self.L + self.L + 1]
        return ventana[:-1], ventana[1:]


dl_lm = DataLoader(VentanasDeTexto(ids_ent, L_LM), batch_size=LOTE_LM,
                   shuffle=True, drop_last=True)

pd.Series({"tokens de entrenamiento": len(ids_ent), "tokens de validación": len(ids_val),
           "vocabulario": V_LM, "pasos por época": len(dl_lm)})

### El modelo de lenguaje de la Clase 2 y su entrenamiento

| Nombre | Qué es |
|---|---|
| `ModeloLenguaje` | `Embedding → LSTM → Linear(V_LM)`, con logits en **cada** posición de la secuencia |
| `perdida_lm` | la entropía cruzada sobre las `B · L` predicciones del lote |
| `perplejidad` | su exponencial, ventana por ventana, sobre una secuencia entera de índices |
| `muestrear` | genera tokens sorteando de la *softmax*, realimentando el token y el estado |
| `entrenar_pasos` | el bucle de entrenamiento de la Clase 2, cortado en una cantidad fija de pasos |
| `DIM_LM`, `OCULTA_LM`, `LR_LM`, `PASOS`, `CADA_LM` | 128, 256, 2e-3, 1.000 y 250 |

Una época sobre este corpus son 1.016 pasos, así que los 1.000 pasos de `PASOS` son aproximadamente una época. El modelo de la Clase 2 se entrenó cinco.

In [ ]:
# ─── Setup: el modelo de lenguaje de la Clase 2 y su entrenamiento ──────────
DIM_LM, OCULTA_LM = 128, 256
LR_LM, PASOS, CADA_LM = 2e-3, 1_000, 250


class ModeloLenguaje(nn.Module):
    """Embedding → LSTM → Linear. Devuelve V logits EN CADA POSICIÓN de la secuencia."""

    def __init__(self, n_vocab=None, dim_emb=DIM_LM, dim_oculta=OCULTA_LM):
        super().__init__()
        n_vocab = n_vocab or V_LM
        self.embedding = nn.Embedding(n_vocab, dim_emb)
        self.lstm = nn.LSTM(dim_emb, dim_oculta, batch_first=True)
        self.salida = nn.Linear(dim_oculta, n_vocab)

    def forward(self, x, estado=None):
        h, estado = self.lstm(self.embedding(x), estado)
        return self.salida(h), estado


def perdida_lm(logits, y):
    """Entropía cruzada sobre las B·L predicciones del lote."""
    return F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1))


@torch.no_grad()
def perplejidad(m, ids, L=None, lote=256):
    """Exponencial de la entropía cruzada media, ventana por ventana (Clase 2, sección 3.1)."""
    L = L or L_LM
    m.eval()
    nll, n = 0.0, 0
    for x, y in DataLoader(VentanasDeTexto(ids, L), batch_size=lote):
        logits, _ = m(x.to(DISPOSITIVO))
        nll += perdida_lm(logits, y.to(DISPOSITIVO)).item() * y.numel()
        n += y.numel()
    m.train()
    return math.exp(nll / n)


@torch.no_grad()
def muestrear(m, semilla, n=20, seed=0):
    """Genera n tokens sorteando de la softmax, realimentando el token y el estado."""
    m.eval()
    gen = torch.Generator().manual_seed(seed)
    ids = [vocab_lm[t] for t in semilla.split()]
    logits, estado = m(torch.tensor([ids], device=DISPOSITIVO))
    for _ in range(n):
        p = torch.softmax(logits[0, -1], -1).cpu()
        siguiente = torch.multinomial(p, 1, generator=gen).item()
        ids.append(siguiente)
        logits, estado = m(torch.tensor([[siguiente]], device=DISPOSITIVO), estado)
    m.train()
    return " ".join(vocab_lm.decodificar(ids))


def entrenar_pasos(m, pasos=PASOS):
    """El bucle de entrenamiento de la Clase 2, cortado en una cantidad fija de pasos."""
    opt = torch.optim.Adam(m.parameters(), lr=LR_LM)
    historia, paso, acumulada = [], 0, 0.0
    while paso < pasos:
        for x, y in dl_lm:
            x, y = x.to(DISPOSITIVO), y.to(DISPOSITIVO)
            logits, _ = m(x)
            perdida = perdida_lm(logits, y)
            opt.zero_grad()
            perdida.backward()
            nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step()
            paso += 1
            acumulada += perdida.item()
            if paso % CADA_LM == 0:
                historia.append({"paso": paso, "ppl_ent": math.exp(acumulada / CADA_LM)})
                print(f"   paso {paso:5d}  ppl ent {historia[-1]['ppl_ent']:8.2f}", flush=True)
                acumulada = 0.0
            if paso >= pasos:
                break
    return {"historia": historia}


print(f"modelo de lenguaje de la clase: {n_parametros(ModeloLenguaje()):,} parámetros")

### Ejercicio 4 — El mismo cambio, sobre el modelo de lenguaje

**Objetivo:** Aplicarle al modelo de lenguaje de la Clase 2 el mismo cambio de la Sección A y medir su perplejidad en validación.

**Enunciado:**

1. **Escribí la clase `ModeloLenguajeBidireccional`**, que herede de `ModeloLenguaje`. Cambian dos cosas: la LSTM pasa a ser bidireccional con `OCULTA_LM` unidades **por dirección** —acá no hace falta partir el estado al medio, porque no hay ningún decoder esperando un tamaño—, y la `Linear` de salida pasa a recibir las dos direcciones concatenadas, que es lo que le llega en cada posición. El `forward` que hereda no se toca.

> **Nota — cuál de las dos salidas de la capa.** Una `nn.LSTM` devuelve dos cosas, y con `bidirectional=True` cada una junta las direcciones de manera distinta. La salida **en cada posición** tiene forma `(B, L, 2 * oculta)`: las dos direcciones vienen concatenadas en la última dimensión, token por token, y esa concatenación ya está hecha. El **estado final** `(h_n, c_n)` tiene forma `(2, B, oculta)` cada uno: las direcciones vienen apiladas en un eje nuevo, y juntarlas es trabajo de quien las use. En la Sección A el decoder necesitaba el estado final, y de ahí salieron el `permute` y el `reshape` del Ejercicio 1; acá el `forward` de `ModeloLenguaje` le pasa a la `Linear` la salida en cada posición, así que no hay nada que reacomodar y alcanza con ampliarla a `2 * dim_oculta`. Es la misma información presentada de dos maneras, según lo que se le pida a la capa.
>
> Cuidado con la letra, que en las dos secciones es la misma y el objeto no: el `h` de `ModeloLenguaje.forward` —y también el de `Traductor.forward`— es la salida en cada posición, mientras que el `h` que devolviste en `codificar` en el Ejercicio 1 es el estado final. Los nombres vienen de los notebooks de teoría y quedan como están; lo que hay que tener presente es de qué objeto habla cada celda.

2. **Entrenalo 1.000 pasos.** Usá `entrenar_o_cargar` con el nombre `"lab2_lm_bidi"`, `vocab_lm`, un constructor que fije la semilla en 0, y `entrenar_pasos` como entrenador. En la configuración declará `arq="ModeloLenguajeBidireccional"`, `L`, `dim`, `oculta`, `pasos`, `lote`, `lr`, `n_ent=len(ids_ent)` y `seed=0`. El modelo entrenado tiene que quedar en una variable llamada `lm_bidi`, que el Ejercicio 5 vuelve a usar.

3. **Medí la perplejidad en validación** con `perplejidad` sobre `ids_val`, e imprimila junto a la cantidad de parámetros.

> **Nota:** la Clase 2 midió, sobre esta misma validación, **73,1** de perplejidad para el modelo de lenguaje entrenado cinco épocas, **100,8** para el trigrama interpolado y **432,6** para el unigrama por conteo. El vocabulario tiene 5.003 tokens, así que un modelo que sortease al azar daría 5.003.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

a) La perplejidad es la cantidad de tokens entre los que el modelo duda, en promedio, antes de elegir el siguiente (Clase 2, sección *3.1*). Traducí el valor que te dio a esa lectura, en una oración.

b) Poné ese valor al lado de los tres de la nota y escribí, **antes** de hacer el Ejercicio 5, una hipótesis sobre qué lo produce. No hace falta que aciertes: lo que se corrige es que la hipótesis sea comprobable, es decir, que digas qué medición la confirmaría o la descartaría.

*(Escribí tu respuesta acá)*

### Ejercicio 5 — De dónde sale ese número

**Objetivo:** Localizar con una medición propia de dónde sale el valor de perplejidad del ejercicio anterior.

**Enunciado:**

La perplejidad que mediste es un promedio sobre las 32 posiciones de cada ventana. En esta tarea hay una posición que se distingue de las demás por construcción, y es la última de cada ventana, la única que no tiene ninguna palabra después dentro de la ventana.

**Parte A — La medición.**

1. **Escribí la función `perplejidad_por_posicion(m, ids, L=L_LM, lote=256)`**, que devuelva un tensor de `L` valores, la perplejidad promedio en cada posición de la ventana. Es la misma cuenta que `perplejidad`, sin promediar sobre las posiciones: la entropía cruzada de cada predicción por separado, promediada entre las ventanas, y recién al final la exponencial. Imprimí las cinco primeras posiciones y la última.

2. **Medí el acierto exacto.** Tomá un lote de 64 ventanas de validación, pedile al modelo su predicción más probable en cada posición, y calculá qué proporción de las posiciones **que tienen una palabra después dentro de la ventana** acierta exactamente el token siguiente, y qué proporción acierta en la **última** posición. Imprimí las dos.

> **Nota:** `F.cross_entropy(..., reduction="none")` devuelve la entropía cruzada de cada predicción en vez del promedio, con una entrada por posición del lote. Eso es lo que hay que ir acumulando por posición.

> **Pista:** la predicción más probable en cada posición es `logits.argmax(-1)`, que tiene la misma forma `(B, L)` que `y`.

In [ ]:
# Tu código aquí

**Parte B — Generar.**

3. Generá 30 tokens con `muestrear` a partir de `"la princesa"` e imprimilos.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

a) Explicá qué está haciendo el modelo en las posiciones donde la perplejidad es más baja, nombrando **exactamente** qué parte de la entrada usa para producir esa predicción, y por qué la última posición de la ventana es la única que se comporta distinto.

b) Poné el texto que generaste en la Parte B al lado de la perplejidad que mediste en el Ejercicio 4. Explicá qué cambia entre las dos situaciones para que un mismo modelo dé esos dos resultados.

*(Escribí tu respuesta acá)*

### Ejercicio 6 — La misma línea, dos tareas

**Objetivo:** Explicar qué diferencia entre las dos tareas produce los dos resultados del laboratorio.

**Enunciado:**

En las dos secciones agregaste una segunda pasada en sentido contrario, con el mismo parámetro, sobre la misma clase de capa y sobre texto en español. En la Sección A el BLEU subió en los cuatro tramos de largo. En la Sección B, el modelo alcanzó una perplejidad de 1,19 copiando de la entrada la palabra que tenía que predecir, y no pudo generar texto.

a) Explicá qué propiedad de cada una de las dos tareas produce esa diferencia. La respuesta tiene que decir, para cada tarea, **qué tiene disponible el modelo en el momento en que produce cada predicción**, y por qué en una de las dos la palabra a predecir está del lado de la entrada.

b) Proponé un cambio **en la tarea** —no en el modelo— que permita usar las dos direcciones sin que la respuesta esté a la vista, y explicá por qué el modelo resultante seguiría sin poder generar texto.

*(Escribí tu respuesta acá)*

---
## Antes de entregar

Revisá esta checklist rápida:

- [ ] Reinicié el entorno y ejecuté **todas** las celdas de arriba a abajo sin errores (**Entorno de ejecución > Reiniciar y ejecutar todo**).
- [ ] Las dos comparaciones del punto 4 del Ejercicio 1 dan `True`, y están impresas.
- [ ] Las cuatro corridas del Ejercicio 3 tienen sus semillas declaradas en la configuración de `entrenar_o_cargar`, así que son cuatro modelos distintos y no el mismo cargado cuatro veces.
- [ ] La tabla del Ejercicio 2 tiene sus siete filas, y las dos columnas son el traductor de la clase y el bidireccional.
- [ ] El Ejercicio 5 imprime la perplejidad de las cinco primeras posiciones, la de la última, los dos aciertos exactos y el texto generado.
- [ ] Respondí las preguntas de análisis de los Ejercicios 1 a 5 y el análisis final del Ejercicio 6.
- [ ] No modifiqué ninguna celda fuera de las de actividad.

---
## ¡Listo!

Hiciste dos veces el mismo cambio de una línea y obtuviste dos resultados opuestos. En traducción, el encoder bidireccional subió el BLEU en los cuatro tramos de largo con menos parámetros que el traductor de la clase, y volvió innecesaria la inversión de la entrada. En modelado de lenguaje, el mismo cambio produjo un modelo que acierta el 99,9 % de las posiciones que tienen una palabra después dentro de la ventana y que no puede generar texto.

Entre las dos quedó la práctica central del laboratorio, que es **verificar de dónde sale un número antes de darlo por bueno**. La perplejidad 1,19 no se descartó por intuición sino con dos mediciones —la perplejidad por posición y el acierto exacto—, y la mejora de traducción no se aceptó por ser grande sino después de medir cuánto se mueve el BLEU entre corridas idénticas salvo por la semilla.

Lo que sigue en la **Parte B** de este laboratorio es la tercera idea de la propuesta: qué pasa si las dos tablas de palabras del modelo de lenguaje —la de entrada y la de salida— son una sola.

Otras tres cuestiones quedaron afuera del laboratorio y conviene nombrarlas:

- **Otras formas de juntar los dos estados finales.** Sumarlos, o proyectar los 512 números concatenados a 256 con una `Linear`. Las dos usan `OCULTA` por dirección en vez de la mitad, así que agregan parámetros; en las corridas de la cátedra las dos quedaron por debajo de la concatenación.
- **La atención de la Clase 4**, que aborda el mismo problema desde el otro lado: en vez de mejorar el resumen, deja de usar un resumen. En las corridas de esa clase, la atención llevó el BLEU de 26,2 a 31,2 sobre este mismo conjunto de prueba. Poner ese salto al lado del que mediste vos pide entrenar los dos en la misma máquina, que es exactamente lo que practicaste en el Ejercicio 3. Lo que sí se puede decir es que las dos ideas son compatibles, porque un encoder bidireccional combinado con atención es el modelo del artículo original de Bahdanau.
- **La forma de usar las dos direcciones en un modelo de lenguaje sin que la respuesta esté en la entrada**, que es la que propusiste en el Ejercicio 6. Esa idea tiene nombre y es la base de una familia entera de modelos, que la Unidad 4 va a tratar.

Lo que sigue, en la Unidad 3, reemplaza la recurrencia por un mecanismo que calcula la representación de cada posición en función de todas las demás, de a una sola operación. Cuando ese mecanismo se usa para predecir la palabra siguiente hay que impedirle mirar hacia adelante, y eso se hace con una **máscara causal**, que es exactamente el problema de la Sección B resuelto de otra manera.